![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/header_workshop.png)

   

 Item | Descrição |
 --- | --- |
 **Objetivo** | Criar diferentes Funções no Unity Catalog |
 **Databricks Run Time** | DBR 16.4 LTS |
 **Linguagem** | Python, Pyspark e SQL |

![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/img/02_diagram.png)

   
## Criar Funções

   
![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/img/02_functions.png)

**Funções SQL** são projetadas para simplificar e otimizar o processamento de dados em consultas, permitindo que os usuários encapsulem lógica reutilizável para cálculos, transformações de valores ou agregações de resultados diretamente em instruções SQL; no Databricks, essas funções são governadas pelo Unity Catalog, garantindo gerenciamento centralizado e seguro, além de governança consistente em todos os ativos.

   
### Buscar Loja por ID

In [0]:
%sql
    
CREATE OR REPLACE FUNCTION serverless_stable_bettanim_catalog.workshop_ml_agentes.get_store_by_id(
  p_store_id STRING COMMENT 'O ID único da loja a ser pesquisada.'
)
RETURNS TABLE(
  store STRING COMMENT 'Nome ou identificador da loja.',
  store_type STRING COMMENT 'Tipo ou categoria da loja.',
  store_zip STRING COMMENT 'CEP da loja.',
  retailer STRING COMMENT 'Nome do varejista ao qual a loja pertence.',
  store_lat_long STRING COMMENT 'Coordenadas de latitude e longitude da loja.'
)
COMMENT 'Função para recuperar dados da loja com base no ID fornecido. Retorna informações da metric view mvw_inventory.'
RETURN
  SELECT
    store,
    store_type,
    store_zip,
    retailer,
    store_lat_long
  FROM
    serverless_stable_bettanim_catalog.workshop_ml_agentes.dim_store
  WHERE store_id = p_store_id
  LIMIT 1;

In [0]:
%sql
SELECT * FROM serverless_stable_bettanim_catalog.workshop_ml_agentes.get_store_by_id("4823349147981451781")

   
### Calcular Latitude e Longitude

In [0]:
%sql
    
CREATE OR REPLACE FUNCTION serverless_stable_bettanim_catalog.workshop_ml_agentes.geocode_address(address STRING)
RETURNS STRUCT<latitude DOUBLE, longitude DOUBLE>
LANGUAGE PYTHON
AS
$$
import http.client
import json
import urllib.parse

def geocode_via_nominatim(address):
    if not address:
        return (None, None)
        
    encoded_address = urllib.parse.quote(address)
    
    conn = http.client.HTTPSConnection('nominatim.openstreetmap.org')
    
    headers = { 
        'User-Agent': f'Databricks-Geocoding-Function/1.0',
        'Accept': 'application/json'
    } 
    
    # Caminho da requisição com f-string corrigida
    request_path = f'/search?q={encoded_address}&format=json&limit=1'
    
    try:
        conn.request('GET', request_path, headers=headers)
        
        response = conn.getresponse()
        
        data = json.loads(response.read().decode())
        
        if data and len(data) > 0:
            return (float(data[0]['lat']), float(data[0]['lon']))
        else:
            return (None, None)
    except Exception:
        return (None, None)
    finally:
        conn.close()

result = geocode_via_nominatim(address)
return {'latitude': result[0], 'longitude': result[1]}
$$;

In [0]:
%sql
WITH geocoded_location AS (
  SELECT 
    serverless_stable_bettanim_catalog.workshop_ml_agentes.geocode_address('Rua Sete de Setembro, 66, Rio de Janeiro, Brazil') AS coords
)
SELECT
  coords.latitude,
  coords.longitude
FROM
  geocoded_location;